In [ ]:
!pip install transformers torch pandas scikit-learn captum

In [ ]:
import torch
print(torch.cuda.is_available())       # Should print True
print(torch.cuda.get_device_name(0))   # Should print Tesla T4

True
Tesla T4


In [6]:
from google.colab import files
import pandas as pd

# ── Upload and auto-detect ────────────────────────────
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f"Using file: {CSV_PATH}")

# ── Redefine variables ────────────────────────────────
TEXT_COLUMN       = "sentence"
LABEL_COLUMNS     = ["gender_sensitive", "stereotyping", "representation"]
AUTHORSHIP_COLUMN = "authorship"

df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")

print("\n── Label Distribution ──────────────────────────────")
for label in LABEL_COLUMNS + [AUTHORSHIP_COLUMN]:
    count = df[label].sum()
    total = len(df)
    pct   = count / total * 100
    bar   = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
    print(f"  {label:<20} {bar} {count}/{total} ({pct:.1f}%)")

print("\n── Multi-label Sentences ───────────────────────────")
all_labels     = LABEL_COLUMNS + [AUTHORSHIP_COLUMN]
df['n_labels'] = df[all_labels].sum(axis=1)
print(df['n_labels'].value_counts().sort_index().rename({
    0: 'neutral (0 labels)',
    1: 'single label',
    2: 'two labels',
    3: 'three labels',
    4: 'four labels'
}))

Saving dataset_sl.csv to dataset_sl (7).csv
Using file: dataset_sl (7).csv
Total rows: 545

── Label Distribution ──────────────────────────────
  gender_sensitive     █████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 147/545 (27.0%)
  stereotyping         ███████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 165/545 (30.3%)
  representation       ██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 159/545 (29.2%)
  authorship           ███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 121/545 (22.2%)

── Multi-label Sentences ───────────────────────────
n_labels
neutral (0 labels)     80
single label          355
two labels             93
three labels           17
Name: count, dtype: int64


In [7]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_NAME  = "bert-base-uncased"
LABELS      = ["gender_sensitive", "stereotyping", "representation", "authorship"]
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 20          # set high, early stopping will handle it
LR          = 2e-5
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using: {DEVICE}")

# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
# df = pd.read_csv("/content/drive/MyDrive/THESIS/DATASET/dataset_sl.csv")

df = df.dropna(subset=["sentence"])
print(f"Dataset size: {len(df)} rows")
print(df[LABELS].sum())

# ── Dataset Class ─────────────────────────────────────────────────────────────
class SentenceDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len):
        self.sentences = sentences
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.sentences[idx],
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float)
        }

# ── Prepare Data ──────────────────────────────────────────────────────────────
X = df["sentence"].tolist()
y = df[LABELS].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tokenizer     = BertTokenizer.from_pretrained(MODEL_NAME)
train_dataset = SentenceDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = SentenceDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# ── Model ─────────────────────────────────────────────────────────────────────
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels   = len(LABELS),
    problem_type = "multi_label_classification"
)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)

# ── Early Stopping ────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=2, min_delta=0.01):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = float("inf")
        self.counter     = 0
        self.should_stop = False

    def check(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            print(f"  ✓ Val loss improved to {val_loss:.4f}")
        else:
            self.counter += 1
            print(f"  ✗ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.should_stop = True
                print("  ⚠️ Early stopping triggered!")

# ── Train Epoch — no batch printing ──────────────────────────────────────────
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:                          # ← removed batch print
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        output.loss.backward()
        optimizer.step()

        total_loss += output.loss.item()

    return total_loss / len(loader)

# ── Evaluate Loss ─────────────────────────────────────────────────────────────
def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            output      = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += output.loss.item()

    return total_loss / len(loader)

# ── Evaluate Predictions ──────────────────────────────────────────────────────
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            output = model(input_ids=input_ids, attention_mask=attention_mask)
            probs  = torch.sigmoid(output.logits)
            preds  = (probs >= threshold).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return np.vstack(all_preds), np.vstack(all_labels)

# ── Run Training ──────────────────────────────────────────────────────────────
early_stopping = EarlyStopping(patience=2, min_delta=0.01)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer)
    val_loss   = evaluate_loss(model, test_loader)

    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")

    early_stopping.check(val_loss)

    if early_stopping.should_stop:
        print(f"\n  Stopped at epoch {epoch + 1}")
        break

# ── Final Evaluation ──────────────────────────────────────────────────────────
y_pred, y_true = evaluate(model, test_loader)

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

exact_match = np.all(y_true == y_pred, axis=1).mean()
hamming_acc = (y_true == y_pred).mean()
print(f"Exact Match Accuracy: {exact_match:.4f}")
print(f"Hamming Accuracy:     {hamming_acc:.4f}")

# ── Save Model ────────────────────────────────────────────────────────────────
model.save_pretrained("/content/drive/MyDrive/THESIS/bert_gender_model")
tokenizer.save_pretrained("/content/drive/MyDrive/THESIS/bert_gender_model")
print("\nModel saved to Google Drive!")

Using: cuda
Dataset size: 545 rows
gender_sensitive    147
stereotyping        165
representation      159
authorship          121
dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/20
  Train Loss : 0.5941
  Val Loss   : 0.5159
  ✓ Val loss improved to 0.5159

Epoch 2/20
  Train Loss : 0.4559
  Val Loss   : 0.4355
  ✓ Val loss improved to 0.4355

Epoch 3/20
  Train Loss : 0.3715
  Val Loss   : 0.3613
  ✓ Val loss improved to 0.3613

Epoch 4/20
  Train Loss : 0.2935
  Val Loss   : 0.3041
  ✓ Val loss improved to 0.3041

Epoch 5/20
  Train Loss : 0.2271
  Val Loss   : 0.2470
  ✓ Val loss improved to 0.2470

Epoch 6/20
  Train Loss : 0.1795
  Val Loss   : 0.2217
  ✓ Val loss improved to 0.2217

Epoch 7/20
  Train Loss : 0.1373
  Val Loss   : 0.2140
  ✗ No improvement (1/2)

Epoch 8/20
  Train Loss : 0.1035
  Val Loss   : 0.2057
  ✓ Val loss improved to 0.2057

Epoch 9/20
  Train Loss : 0.0735
  Val Loss   : 0.2125
  ✗ No improvement (1/2)

Epoch 10/20
  Train Loss : 0.0573
  Val Loss   : 0.1988
  ✗ No improvement (2/2)
  ⚠️ Early stopping triggered!

  Stopped at epoch 10

--- Classification Report ---
                  precision    recall  f1-score   suppo

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to Google Drive!


In [10]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

LABELS    = ["gender_sensitive", "stereotyping", "representation", "authorship"]
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THRESHOLD = 0.7

# ── Load model with eager attention ──────────────────────────────────────────
model     = BertForSequenceClassification.from_pretrained(
                "/content/drive/MyDrive/THESIS/bert_gender_model",
                attn_implementation = "eager"       # ← fix for sdpa warning
            )
tokenizer = BertTokenizer.from_pretrained(
                "/content/drive/MyDrive/THESIS/bert_gender_model"
            )
model.to(DEVICE)

# ── Trigger Words via Attention ───────────────────────────────────────────────
def get_trigger_words(tokens, attn_scores, top_k=3):
    words       = []
    word_scores = []
    curr_word   = ""
    curr_score  = 0

    for token, score in zip(tokens, attn_scores):
        if token in ("[CLS]", "[SEP]", "[PAD]"):
            continue
        if token.startswith("##"):
            curr_word  += token[2:]
            curr_score  = max(curr_score, float(score))
        else:
            if curr_word:
                words.append(curr_word)
                word_scores.append(curr_score)
            curr_word  = token
            curr_score = float(score)

    if curr_word:
        words.append(curr_word)
        word_scores.append(curr_score)

    ranked = sorted(zip(words, word_scores), key=lambda x: x[1], reverse=True)
    return [word for word, _ in ranked[:top_k]]


# ── Predict Function ──────────────────────────────────────────────────────────
def predict(sentence, threshold=THRESHOLD):
    model.eval()

    encoding = tokenizer(
        sentence,
        max_length     = 128,
        padding        = "max_length",
        truncation     = True,
        return_tensors = "pt"
    )

    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)
    tokens         = tokenizer.convert_ids_to_tokens(input_ids[0])

    with torch.no_grad():
        output = model(
            input_ids         = input_ids,
            attention_mask    = attention_mask,
            output_attentions = True            # ← now works with eager
        )
        probs = torch.sigmoid(output.logits).squeeze(0).cpu().numpy()
        preds = (probs >= threshold).astype(int)

        # ── Average attention across all layers and heads ─────────────────────
        attentions  = torch.stack(output.attentions)
        avg_attn    = attentions.mean(dim=0)
        avg_attn    = avg_attn.mean(dim=1)
        attn_scores = avg_attn.squeeze(0).mean(dim=0).cpu().numpy()

    triggers = get_trigger_words(tokens, attn_scores, top_k=3)

    print(f"\nSentence : {sentence}")
    print(f"\n{'Label':<20} {'Probability':>12} {'Prediction':>12}   {'Trigger Words'}")
    print("-" * 75)

    for label, prob, pred in zip(LABELS, probs, preds):
        if pred == 1:
            print(f"{label:<20} {prob:>12.4f} {'Yes':>12}   → {', '.join(triggers)}")
        else:
            print(f"{label:<20} {prob:>12.4f} {'No':>12}")


# ── Trigger Words via Attention ───────────────────────────────────────────────
IGNORE_TOKENS = {
    "[CLS]", "[SEP]", "[PAD]",
    ".", ",", "!", "?", "'", '"',
    "''", "``", "--", "-", ":",
    ";", "(", ")", "a", "the",
    "is", "are", "was", "were",
    "it", "its", "be", "been",
    "an", "of", "in", "to",
    "and", "or", "not", "for",
    "on", "at", "by", "with"
}

def get_trigger_words(tokens, attn_scores, top_k=3):
    words       = []
    word_scores = []
    curr_word   = ""
    curr_score  = 0

    for token, score in zip(tokens, attn_scores):
        if token in IGNORE_TOKENS:
            continue
        if token.startswith("##"):
            curr_word  += token[2:]
            curr_score  = max(curr_score, float(score))
        else:
            if curr_word:
                words.append(curr_word)
                word_scores.append(curr_score)
            curr_word  = token
            curr_score = float(score)

    if curr_word:
        words.append(curr_word)
        word_scores.append(curr_score)

    ranked = sorted(zip(words, word_scores), key=lambda x: x[1], reverse=True)
    return [word for word, _ in ranked[:top_k]]


# ── Test Sentences ────────────────────────────────────────────────────────────
# gender_sensitive
predict("The stewardess greeted passengers as they boarded the plane.")
predict("The chairman opened the meeting with a brief introduction.")
predict("Every freshman must attend the orientation seminar.")
predict("The mailman delivered the package early in the morning.")
predict("The businessmen gathered for the annual economic summit.")

# stereotyping
predict("Women are naturally more emotional than men in the workplace.")
predict("Men are better suited for leadership roles than women.")
predict("Girls are generally weaker in mathematics compared to boys.")
predict("Fathers are less nurturing than mothers when raising children.")
predict("Women are more suited for caregiving than engineering.")

# representation
predict("All nurses in the study were referred to using female pronouns.")
predict("The research assumed all engineers were male.")
predict("Women were mostly portrayed as passive participants in the study.")
predict("The paper referred to all doctors as he throughout the discussion.")
predict("Male respondents were predominantly chosen for the leadership survey.")

# authorship
predict("Marie Curie is the only woman to have won two Nobel Prizes.")
predict("Sigmund Freud developed the theory of psychoanalysis.")
predict("Isaac Newton formulated the laws of motion and universal gravitation.")
predict("Charles Darwin introduced the theory of evolution by natural selection.")
predict("Stephen Hawking proposed groundbreaking theories on black holes.")

# neutral
predict("The study was conducted over a period of six months.")
predict("Data was collected from a diverse group of respondents.")
predict("The results were analyzed using statistical methods.")
predict("Both groups showed similar patterns in the findings.")
predict("The researchers reviewed existing literature on the topic.")

# multiple
predict("The stewardess was naturally more nurturing than her male colleagues.")        # gender_sensitive + stereotyping
predict("Policewomen are often portrayed as less decisive than policemen.")             # gender_sensitive + representation
predict("The businessmen were always more rational than their female counterparts.")    # gender_sensitive + stereotyping
predict("Women are too emotional to lead, as noted by John Smith in his study.")        # stereotyping + authorship
predict("Marie Curie was an exception, as women are generally weaker in science.")      # authorship + stereotyping
predict("The fireman, unlike female workers, was portrayed as the true hero.")          # gender_sensitive + representation
predict("Robert Johnson found that women are naturally better suited for nursing.")     # authorship + stereotyping
predict("Mankind has always assumed women belong in caregiving roles.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Sentence : The stewardess greeted passengers as they boarded the plane.

Label                 Probability   Prediction   Trigger Words
---------------------------------------------------------------------------
gender_sensitive           0.9798          Yes   → greeted, passengers, stewardess
stereotyping               0.0454           No
representation             0.0348           No
authorship                 0.0300           No

Sentence : The chairman opened the meeting with a brief introduction.

Label                 Probability   Prediction   Trigger Words
---------------------------------------------------------------------------
gender_sensitive           0.9660          Yes   → chairman, opened, introduction
stereotyping               0.0381           No
representation             0.0277           No
authorship                 0.0261           No

Sentence : Every freshman must attend the orientation seminar.

Label                 Probability   Prediction   Trigger Words
-